In [22]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [23]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [14]:
# TODO: Read image data, feed into Claude
with open("images/prop4.png", "rb") as f:
    image_data = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []
add_user_message(messages,[
    # Image Block
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_data,
        }
    },
    # Text Block
    {
        "type": "text",
        "text": prompt,
    }
])

response = chat(messages)

In [15]:
print(response.content[0].text)

# Property Fire Risk Analysis

**1. Residence Identification:**
The primary residence is a large house with multiple roof sections and prominent gables, located in the center of the property with a driveway connection curving from the bottom right, and a detached outbuilding (likely a garage or shed) visible in the upper left portion of the property.

**2. Tree Overhang Analysis:**
Multiple mature trees are positioned immediately adjacent to the residence, with at least one large tree on the northeast corner and another on the southeast corner appearing to have canopy that extends directly over portions of the roof, affecting approximately 15-25% of the total roof area.

**3. Fire Risk Assessment:**
The overhanging trees create potential ember catch points where branches meet the roof surfaces, particularly concerning on the darker-colored roof sections where debris accumulation would be likely, and the trees form a direct fuel pathway that could allow fire to transition from vegetatio

In [8]:
messages

[{'role': 'user',
  'content': [{'type': 'image',
    'source': {'type': 'base64',
     'media_type': 'image/png',
     'data': '...'}},
   {'type': 'text',
    'text': '\nAnalyze the attached satellite image of a property with these specific steps:\n\n1. Residence identification: Locate the primary residence on the property by looking for:\n   - The largest roofed structure \n   - Typical residential features (driveway connection, regular geometry)\n   - Distinction from other structures (garages, sheds, pools)\n   Describe the residence\'s location relative to property boundaries and other features.\n\n2. Tree overhang analysis: Examine all trees near the primary residence:\n   - Identify any trees whose canopy extends directly over any portion of the roof\n   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)\n   - Note particularly dense areas of overhang\n\n3. Fire risk assessment: For any overhanging trees, evaluate:\n   - Potential

### Working with PDFs:

In [24]:
# TODO: Read image data, feed into Claude
with open("earth.pdf", "rb") as f:
    file_data = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []
add_user_message(messages,[
    # Document Block
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_data,
        },
        "title": "earth.pdf",
        "citations": {"enabled": True}
    },
    # Text Block
    {
        "type": "text",
        "text": "What is the Polar radius of the Earth? and its mass.",
    }
])

response = chat(messages)

In [25]:
response

Message(id='msg_01Loe5dcMTZ9H16EgAiqfgNG', container=None, content=[TextBlock(citations=None, text='According to the document, ', type='text'), TextBlock(citations=[CitationPageLocation(cited_text='1.578 69° – invariable\r\nplane;\r\n[4]\r\n0.000 05° – J2000 ecliptic\r\nLongitude of\r\nascending node\r\n−11.260 64° – J2000\r\necliptic\r\n[3]\r\nTime of\r\nperihelion\r\n2023-Jan-04\r\n[5]\r\nArgument of\r\nperihelion\r\n114.207 83°\r\n[3]\r\nSatellites 1, the Moon\r\nPhysical characteristics\r\nMean radius 6 371.0 km\r\n[6]\r\nEquatorial\r\nradius\r\n6 378.137 km\r\n[7][8]\r\nPolar radius 6 356.752 km[9]\r\nFlattening 1/298.257 222 101\r\n(ETRS89)\r\n[10]\r\nCircumference 40 075.017 km equatorial\r\n[8]\r\n40 007.86 km\r\nmeridional\r\n[11][n 3]\r\nSurface area 510 072 000 km2[12][n 4]\r\nLand: 148 940 000 km2\r\nWater: 361 132 000 km2\r\nVolume 1.083 21 × 10\r\n12 km3[3]\r\nMass 5.972 168 × 10\r\n24 kg\r\n[13]\r\nMean density 5.513 g/cm3[3]\r\nSurface gravity 9.806 65 m/s\r\n2[14]\r\n(

In [26]:
text_from_message(response)

'According to the document, \nthe polar radius of Earth is 6,356.752 km\n, and \nits mass is 5.972168 × 10²⁴ kg\n.'

In [27]:
messages

[{'role': 'user',
  'content': [{'type': 'document',
    'source': {'type': 'base64',
     'media_type': 'application/pdf',
     'data': '...'},
    'title': 'earth.pdf',
    'citations': {'enabled': True}},
   {'type': 'text',
    'text': 'What is the Polar radius of the Earth? and its mass.'}]}]